# Étape 2c : génération du dictionnaire

Le dictionnaire est proposé à partir des données observées. Toutes les règles sont au statut `observé`, les écarts sont signalés dans `a_arbitrer`, et les colonnes sensibles n'exposent aucune valeur.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from frigeo import racine_projet
from frigeo.perimetre import decouvrir

# PARAMETRE A MODIFIER CHAQUE MOIS
# Dernier mois déposé dans data/raw, au format AAAA-MM
PERIODE_FIN = "2026-03"

In [ ]:
DATA = racine_projet() / "data"
perimetre = decouvrir(DATA, PERIODE_FIN)
perimetre.verifier()
print(perimetre.resume())

In [ ]:
%%time
from frigeo.chargement import charger_tout
from frigeo.dictionnaire import generer_dictionnaire

tables = charger_tout(perimetre)
dictionnaire = generer_dictionnaire(tables)
print({table: len(colonnes) for table, colonnes in dictionnaire.items()})
print(sum(len(colonnes) for colonnes in dictionnaire.values()), "colonnes")

In [ ]:
from frigeo.dictionnaire import resumer_dictionnaire

resume = resumer_dictionnaire(dictionnaire)
print(resume["liste_fermee"].sum(), "listes fermées proposées sur", len(resume), "colonnes")
display(resume[resume["liste_fermee"]].drop(columns=["motif_liste"]))

In [ ]:
ATTENDUES = {
    ("interventions", "categorie"): {
        "Intervention préventive", "Maintenance", "Dépannage", "Contrôle", "Installation",
    },
    ("interventions", "statut"): {"Réalisée", "Annulée", "Reportée"},
    ("interventions", "facturable"): {"O", "N"},
    ("pieces_interventions", "unite"): {"unité", "kg"},
    ("factures_entetes", "type_piece"): {"Facture", "Avoir"},
    ("factures_entetes", "mode_reglement"): {"Virement", "Chèque", "Prélèvement", "Carte"},
    ("factures_lignes", "code_article"): {
        "MO-STD", "MO-URG", "DEPL", "CTRL-F", "PREV-F", "INST-F", "PIECE", "FLUIDE",
    },
    ("factures_lignes", "unite"): {"heure", "forfait", "unité", "kg"},
    ("achats_entetes", "type_piece"): {"Facture", "Avoir"},
    ("achats_lignes", "unite"): {"unité", "kg", "litre", "forfait"},
    ("clients", "type_commerce"): 8,  # libellés exacts non donnés : comparaison en nombre
    ("clients", "contrat_maintenance"): {"O", "N"},
    ("clients", "statut"): {"Actif", "Inactif"},
    ("fournisseurs", "categorie"): {
        "Pièces frigorifiques", "Fluides", "Carburant", "Frais généraux",
    },
    ("fournisseurs", "delai_paiement"): {"30 j fin de mois", "45 j fin de mois"},
    ("fournisseurs", "statut"): {"Actif", "Inactif"},
    ("catalogue_pieces", "famille"): {
        "Compresseur", "Ventilation", "Régulation", "Joint", "Détente", "Résistance",
        "Sonde", "Fluide",
    },
    ("catalogue_pieces", "unite"): {"unité", "kg"},
    ("intervenants", "poste"): {"Technicien frigoriste", "Chef d'équipe", "Apprenti"},
    ("intervenants", "type_contrat"): {"CDI", "CDD", "Apprentissage"},
    ("intervenants", "motif_sortie"): {
        "Démission", "Fin de CDD", "Rupture conventionnelle", "Retraite",
    },
    ("intervenants", "secteur"): {"27", "76"},
}

lignes = []
for (table, colonne), attendu in ATTENDUES.items():
    valeurs = dictionnaire[table][colonne]["valeurs"]
    proposees = {v["valeur"] for v in valeurs["liste"] if v["origine"] == "liste proposée"}
    a_arbitrer = {v["valeur"] for v in valeurs["liste"] if v["origine"] == "à arbitrer"}
    ligne = {"table": table, "colonne": colonne}
    if valeurs["proposition"] != "liste fermée":
        ligne.update(verdict="non proposée", detail=valeurs["motif"])
    elif isinstance(attendu, int):
        verdict = "identique" if len(proposees) == attendu else "écart"
        ligne.update(verdict=verdict, detail=f"{len(proposees)} valeurs pour {attendu} attendues")
    else:
        manquantes, en_trop = attendu - proposees, proposees - attendu
        if not manquantes and not en_trop:
            verdict = "identique"
        elif manquantes <= a_arbitrer and not en_trop:
            verdict = "valeur rare à arbitrer"
        else:
            verdict = "écart"
        detail = []
        if manquantes:
            detail.append(f"manquantes {sorted(manquantes)}")
        if en_trop:
            detail.append(f"en trop {sorted(en_trop)}")
        ligne.update(verdict=verdict, detail="; ".join(detail))
    lignes.append(ligne)

comparaison = pd.DataFrame(lignes)
display(comparaison["verdict"].value_counts().rename("colonnes").to_frame())
display(comparaison[comparaison["verdict"] != "identique"])

proposees_hors_chapitre = resume[
    resume["liste_fermee"] & ~resume.set_index(["table", "colonne"]).index.isin(list(ATTENDUES))
]
display(proposees_hors_chapitre[["table", "colonne", "nb_valeurs", "nb_a_arbitrer"]])

In [ ]:
# FUSION

from frigeo.dictionnaire import charger_dictionnaire, chemin_dictionnaire
from frigeo.fusion import fusionner

if chemin_dictionnaire().exists():
    genere = dictionnaire
    dictionnaire, rapport = fusionner(charger_dictionnaire(), genere)
    print(len(rapport), "événements, dont", (rapport["niveau"] == "à regarder").sum(), "à regarder")
    print("identique à la génération :", dictionnaire == genere)
    display(rapport)
else:
    print("Premier dictionnaire : rien à fusionner")

## Synthèse de la comparaison avec le chapitre 4

Sur 22 colonnes à liste fermée du chapitre 4 : 13 sont retrouvées à l'identique, 2 avec une valeur rare à arbitrer (`Avoir` dans `type_piece` des factures et des achats, soit 0,13 % et 0,27 % des lignes), et 7 ne sont pas proposées (6 faute d'effectif, car `fournisseurs` et `intervenants` ont moins de 200 lignes; 1 colonne sensible, `motif_sortie`). Aucune valeur en trop, aucun libellé divergent. Trois listes sont proposées hors chapitre 4 : `JournalCode` et `JournalLib` (plausibles), et `fournisseur_principal` (faux positif, clé vers le référentiel des fournisseurs, à invalider par le métier).

## Écriture du dictionnaire

Le dictionnaire est écrit dans `data/config/dictionnaire.yaml`, hors du dépôt, car il contient des valeurs observées dans les données. L'écriture est refusée si une décision de revue du fichier en place serait perdue ou modifiée : une régénération passe donc toujours par la fusion. Un fichier qui porte des décisions est copié dans `data/config/sauvegardes/` avant d'être remplacé.

In [ ]:
# ECRITURE APRES FUSION

from frigeo.dictionnaire import (
    charger_dictionnaire,
    charger_meta,
    dossier_sauvegardes,
    ecrire_dictionnaire,
)
from frigeo.confidentialite import charger_sensibilite

# Sans risque pour les décisions de revue : l'écriture est refusée si l'une d'elles
# serait perdue ou modifiée, et le fichier en place est copié avant d'être remplacé.
chemin = ecrire_dictionnaire(dictionnaire, PERIODE_FIN)
relu = charger_dictionnaire(chemin)

regles = [
    (table, colonne, nom, regle)
    for table, colonnes in relu.items()
    for colonne, lot in colonnes.items()
    for nom, regle in lot.items()
]
valeurs = [valeur for *_, regle in regles for valeur in regle.get("liste", [])]
print(chemin)
print(len(relu), "tables,", sum(len(c) for c in relu.values()), "colonnes,",
      len(regles), "règles,", len(valeurs), "valeurs")
print("statuts :", sorted({element["statut"] for element in [r for *_, r in regles] + valeurs}))
print("décisions de revue :", sum(element["statut"] != "observé" for element in [r for *_, r in regles] + valeurs))
print("identique au dictionnaire fusionné :", relu == dictionnaire)
print("revues au journal :", len(charger_meta()["revues"]))

sensibles = charger_sensibilite()
exposees = [
    (table, colonne)
    for table, colonnes in sensibles.items()
    for colonne in colonnes
    if relu[table][colonne]["valeurs"]["regle"] != "aucune"
    or relu[table][colonne]["valeurs"]["liste"]
]
print(sum(len(c) for c in sensibles.values()), "colonnes sensibles, dont", len(exposees), "exposant des valeurs")
print(round(chemin.stat().st_size / 1024), "Ko")

copies = sorted(dossier_sauvegardes().glob("*.yaml")) if dossier_sauvegardes().exists() else []
print(len(copies), "copies de sauvegarde, dernière :", copies[-1].name if copies else "aucune")

In [ ]:
# EXPORT

from openpyxl import load_workbook

from frigeo.dictionnaire import charger_dictionnaire, charger_meta
from frigeo.revue import exporter_revue

# Le classeur part toujours du dictionnaire enregistré, avec ses décisions.
dictionnaire_courant = charger_dictionnaire()
nb_revues = len(charger_meta()["revues"])
chemin_classeur = exporter_revue(
    dictionnaire_courant, PERIODE_FIN, nb_revues=nb_revues, ecraser=False
)
print(chemin_classeur)
print("revues déjà appliquées :", nb_revues)
classeur = load_workbook(chemin_classeur, read_only=True)
for nom in ("regles", "valeurs"):
    lignes = classeur[nom].iter_rows(min_row=2, values_only=True)
    print(nom, sum(any(v is not None for v in ligne) for ligne in lignes), "lignes")
classeur.close()

In [ ]:
# IMPORT

from frigeo.dictionnaire import charger_dictionnaire, charger_meta
from frigeo.revue import chemin_revue, importer_revue

dictionnaire_courant = charger_dictionnaire()
nb_revues = len(charger_meta()["revues"])
chemin_classeur = chemin_revue(PERIODE_FIN)
revue = importer_revue(chemin_classeur, dictionnaire_courant, nb_revues=nb_revues)
print(len(revue.regles), "règles,", len(revue.valeurs), "valeurs")
print("statuts :", sorted(set(revue.regles["statut"]) | set(revue.valeurs["statut"])))
print("remplacements dans le classeur :", int(revue.valeurs["remplacement"].notna().sum()))
print("avertissements :", revue.avertissements)

In [ ]:
# APPLICATION

from frigeo.revue import appliquer_revue

dictionnaire_revu, rapport_revue, entree_revue = appliquer_revue(
    dictionnaire_courant, revue, chemin_classeur
)
if entree_revue is None:
    print("Aucun changement : rien à écrire.")
else:
    print(entree_revue)
    print(rapport_revue["evenement"].value_counts().to_string())
rapport_revue

In [ ]:
# ECRITURE

from frigeo.dictionnaire import (
    charger_dictionnaire,
    charger_meta,
    dossier_sauvegardes,
    ecrire_dictionnaire,
)

if entree_revue is None:
    print("Aucun changement : dictionnaire non réécrit.")
else:
    # Le fichier en place est copié dans data/config/sauvegardes avant d'être remplacé.
    # Seules les décisions datées de cette revue peuvent changer.
    chemin_ecrit = ecrire_dictionnaire(dictionnaire_revu, PERIODE_FIN, revue=entree_revue)
    assert charger_dictionnaire() == dictionnaire_revu
    print(chemin_ecrit)
    print("revues au journal :", len(charger_meta()["revues"]))
    copies = sorted(dossier_sauvegardes().glob("*.yaml"))
    print(len(copies), "copies de sauvegarde, dernière :", copies[-1].name)
    # Une seconde exécution de la cellule ne doit pas journaliser la revue deux fois.
    entree_revue = None